# Demo Alpha-Gal Serology Workflow


## Load Packages and Demo Data


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

ROOT = Path.cwd()
if not (ROOT / "demo_data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "demo_data" / "simulated_alpha_gal_serology_demo.csv"
TABLE_DIR = ROOT / "outputs" / "tables"
FIGURE_DIR = ROOT / "outputs" / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(DATA_PATH, dtype={"patient_id": str, "accession_id": str, "result_code": str})
data["specimen_collection_date"] = pd.to_datetime(data["specimen_collection_date"])
data.head()

## Basic Data Inspection


In [ ]:
print(f"Dataset shape: {data.shape}")
print("\nColumn names:")
print(list(data.columns))

assay_counts = (
    data.groupby(["analyte_name", "result_code"], as_index=False)
    .size()
    .rename(columns={"size": "record_count"})
    .sort_values(["analyte_name", "result_code"])
)

print("\nAssay counts:")
display(assay_counts)

safe_preview_cols = [
    "patient_id",
    "accession_id",
    "year_of_birth",
    "specimen_collection_date",
    "analyte_name",
    "result_code",
    "reported_result",
    "ige_value_kU_L",
    "unit",
]
display(data[safe_preview_cols].head(8))

## IgE Classification


In [ ]:
def classify_ige(value):
    if value < 0.10:
        return "negative"
    if value < 0.35:
        return "borderline"
    if value < 0.70:
        return "low positive"
    if value < 3.50:
        return "moderate positive"
    return "high positive"


category_order = ["negative", "borderline", "low positive", "moderate positive", "high positive"]
data["ige_category"] = pd.Categorical(
    data["ige_value_kU_L"].apply(classify_ige),
    categories=category_order,
    ordered=True,
)
data["is_positive"] = data["ige_value_kU_L"] >= 0.35

display(data[["analyte_name", "ige_value_kU_L", "ige_category", "is_positive"]].head(10))

## Descriptive Summary


In [ ]:
alpha = data.loc[data["result_code"] == "30039"].copy()
alpha_summary = pd.DataFrame(
    [
        {
            "assay": "Alpha-Gal",
            "records": len(alpha),
            "unique_patients": alpha["patient_id"].nunique(),
            "positive_records": int(alpha["is_positive"].sum()),
            "positivity_percent": round(100 * alpha["is_positive"].mean(), 2),
            "median_ige_kU_L": round(float(alpha["ige_value_kU_L"].median()), 3),
        }
    ]
)

category_counts = (
    data.groupby(["analyte_name", "ige_category"], observed=False)
    .size()
    .reset_index(name="record_count")
)

assay_counts.to_csv(TABLE_DIR / "demo_notebook_record_counts_by_assay.csv", index=False)
alpha_summary.to_csv(TABLE_DIR / "demo_notebook_alpha_gal_positivity_summary.csv", index=False)
category_counts.to_csv(TABLE_DIR / "demo_notebook_ige_category_counts.csv", index=False)

display(alpha_summary)
display(category_counts)

## Paired Same-Day Analysis


In [ ]:
MEAT_ASSAYS = {
    "Beef": "50510S",
    "Pork": "52310S",
    "Lamb/Mutton": "54310S",
}


def make_same_day_pairs(df, meat_name, meat_code):
    key_cols = ["patient_id", "specimen_collection_date"]
    alpha_pairs = (
        df.loc[df["result_code"] == "30039", key_cols + ["ige_value_kU_L"]]
        .rename(columns={"ige_value_kU_L": "alpha_gal_ige_kU_L"})
        .groupby(key_cols, as_index=False)
        .mean(numeric_only=True)
    )
    meat_col = f"{meat_name.lower().replace('/', '_')}_ige_kU_L"
    meat_pairs = (
        df.loc[df["result_code"] == meat_code, key_cols + ["ige_value_kU_L"]]
        .rename(columns={"ige_value_kU_L": meat_col})
        .groupby(key_cols, as_index=False)
        .mean(numeric_only=True)
    )
    paired = alpha_pairs.merge(meat_pairs, on=key_cols, how="inner")
    paired.insert(2, "paired_assay", meat_name)
    return paired, meat_col


correlation_rows = []
paired_tables = {}

for meat_name, meat_code in MEAT_ASSAYS.items():
    paired, meat_col = make_same_day_pairs(data, meat_name, meat_code)
    paired_tables[meat_name] = paired
    rho, p_value = spearmanr(np.log10(paired["alpha_gal_ige_kU_L"]), np.log10(paired[meat_col]))
    correlation_rows.append(
        {
            "paired_assay": meat_name,
            "n": len(paired),
            "spearman_rho_log10_ige": round(float(rho), 4),
            "p_value": float(p_value),
        }
    )
    paired.to_csv(TABLE_DIR / f"demo_notebook_same_day_alpha_gal_vs_{meat_name.lower().replace('/', '_')}.csv", index=False)

correlation_summary = pd.DataFrame(correlation_rows)
correlation_summary.to_csv(TABLE_DIR / "demo_notebook_same_day_spearman_correlations.csv", index=False)
display(correlation_summary)

## Simple Example Figure


In [ ]:
alpha_category_counts = (
    alpha.groupby("ige_category", observed=False)
    .size()
    .reindex(category_order)
    .reset_index(name="record_count")
)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=alpha_category_counts, x="ige_category", y="record_count", color="#4C78A8")
plt.xlabel("IgE category")
plt.ylabel("Simulated alpha-gal records")
plt.title("Simulated alpha-gal IgE category counts")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
figure_path = FIGURE_DIR / "demo_notebook_alpha_gal_ige_category_counts.png"
plt.savefig(figure_path, dpi=200)
plt.show()

print(f"Saved figure to: {figure_path}")

## Final Note
